# 09b — PyTorch Consolidation

Corresponds to: Note 09 — Deep Feedforward Network Architectures

**Libraries introduced in this notebook:** `torch` — PyTorch replaces numpy for all neural network computation from Note 09 onwards. It provides automatic differentiation, GPU support, and `nn.Module` for building networks. Students have already implemented every operation by hand in numpy (Notes 03–08); this notebook shows the same operations in PyTorch side by side.

**What this notebook demonstrates:**
1. Perceptron in PyTorch — forward pass and step activation (Note 02)
2. 2-layer MLP forward pass in PyTorch — XOR network (Note 03)
3. Activation functions via `torch` — sigmoid, tanh, ReLU (Note 04)
4. Loss functions via `torch.nn` — MSE, BCE, cross-entropy (Note 05)
5. Training loop — backprop and gradient descent in PyTorch (Note 06)
6. L-layer deep feedforward network — generalisation to arbitrary depth (Note 09)

In [ ]:
import torch
import torch.nn as nn

print(f'PyTorch version: {torch.__version__}')

---
## Part 1: Perceptron (Note 02)

The perceptron computes z = w·x + b, then applies a step function.

In numpy we wrote this as a scalar loop. In PyTorch, `torch.dot` replaces the dot product and tensors replace arrays. The arithmetic is identical — only the container changes.

In [ ]:
# OR perceptron from Note 02
w = torch.tensor([1.0, 1.0])
b = torch.tensor(-0.5)

def step(z):
    return (z >= 0).float()

inputs = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
targets = torch.tensor([0., 1., 1., 1.])   # OR truth table

print('Perceptron — OR gate:')
print(f'  {"x":>8}   {"z":>6}   {"y_hat":>6}   {"target":>7}')
print(f'  {"-"*8}   {"-"*6}   {"-"*6}   {"-"*7}')
for x, t in zip(inputs, targets):
    z = torch.dot(w, x) + b
    y_hat = step(z)
    print(f'  {str(x.tolist()):>8}   {z.item():>6.2f}   {y_hat.item():>6.0f}   {t.item():>7.0f}')

---
## Part 2: 2-Layer MLP Forward Pass (Note 03)

The XOR network from Note 03: 2 inputs → 2 hidden neurons → 1 output.

In numpy we wrote `W1.T @ x + b1`. In PyTorch, `nn.Linear(in, out)` encapsulates the weight matrix and bias. The forward pass is `layer(x)` — same arithmetic, framework handles the transpose internally.

We set the weights manually to match the Note 03 values so outputs can be verified.

In [ ]:
# XOR network — weights from Note 03
# h1: OR boundary   w=[1,1],  b=-0.5
# h2: NAND boundary w=[-1,-1], b=1.5
# out: AND          w=[1,1],  b=-1.5

layer1 = nn.Linear(2, 2, bias=True)
layer2 = nn.Linear(2, 1, bias=True)

with torch.no_grad():
    layer1.weight.copy_(torch.tensor([[ 1.,  1.],
                                      [-1., -1.]]))
    layer1.bias.copy_(torch.tensor([-0.5, 1.5]))
    layer2.weight.copy_(torch.tensor([[1., 1.]]))
    layer2.bias.copy_(torch.tensor([-1.5]))

xor_inputs  = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
xor_targets = torch.tensor([0., 1., 1., 0.])

print('2-layer MLP — XOR forward pass (Heaviside activation):')
print(f'  {"x":>8}   {"h1":>5}   {"h2":>5}   {"y_hat":>6}   {"target":>7}')
print(f'  {"-"*8}   {"-"*5}   {"-"*5}   {"-"*6}   {"-"*7}')
for x, t in zip(xor_inputs, xor_targets):
    z1 = layer1(x)
    h  = step(z1)
    z2 = layer2(h)
    y_hat = step(z2)
    print(f'  {str(x.tolist()):>8}   {h[0].item():>5.0f}   {h[1].item():>5.0f}   {y_hat.item():>6.0f}   {t.item():>7.0f}')

---
## Part 3: Activation Functions (Note 04)

In numpy we implemented each activation by hand:
- sigmoid: `1 / (1 + np.exp(-z))`
- tanh: `np.tanh(z)`
- ReLU: `np.maximum(0, z)`

PyTorch provides these as `torch.sigmoid`, `torch.tanh`, and `torch.relu` — or as `nn.Sigmoid()`, `nn.Tanh()`, `nn.ReLU()` when used inside a model. The outputs are identical.

In [ ]:
z = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

print('Activation functions applied to z =', z.tolist())
print()
print(f'  Sigmoid:    {torch.sigmoid(z).round(decimals=4).tolist()}')
print(f'  Tanh:       {torch.tanh(z).round(decimals=4).tolist()}')
print(f'  ReLU:       {torch.relu(z).tolist()}')
print(f'  Leaky ReLU: {torch.where(z >= 0, z, 0.01 * z).tolist()}')
print()
print('Softmax (treats z as logits for 5 classes):')
print(f'  {torch.softmax(z, dim=0).round(decimals=4).tolist()}')
print(f'  Sum: {torch.softmax(z, dim=0).sum().item():.6f}  (should be 1.0)')

---
## Part 4: Loss Functions (Note 05)

In numpy we computed each loss by hand from its formula. PyTorch provides them in `torch.nn`.

We verify each one against the Note 05 worked example values.

In [ ]:
# MSE — regression
mse = nn.MSELoss()
y_hat = torch.tensor([2.5, 0.0, 2.0])
y     = torch.tensor([3.0, 0.5, 2.0])
print(f'MSE loss: {mse(y_hat, y).item():.4f}')
print(f'  Manual: {((y_hat - y)**2).mean().item():.4f}  (should match)')
print()

# Binary Cross-Entropy — binary classification
bce = nn.BCELoss()
y_hat_b = torch.tensor([0.9, 0.2, 0.8])
y_b     = torch.tensor([1.0, 0.0, 1.0])
print(f'BCE loss: {bce(y_hat_b, y_b).item():.4f}')
print()

# Categorical Cross-Entropy — multiclass
# nn.CrossEntropyLoss expects raw logits (not softmax), integer class labels
ce = nn.CrossEntropyLoss()
logits = torch.tensor([[2.0, 1.0, 0.1]])   # 1 sample, 3 classes
label  = torch.tensor([0])                  # true class = 0
print(f'Cross-entropy loss: {ce(logits, label).item():.4f}')
print(f'  (applies softmax internally — do not pass softmax output to CrossEntropyLoss)')

---
## Part 5: Training Loop — Backprop and Gradient Descent (Note 06)

In numpy we computed every gradient by hand using the chain rule. PyTorch's autograd does this automatically via `loss.backward()`.

The four steps from Note 03 Part 4 map directly to PyTorch:

```
1. Forward pass:      y_hat = model(x)
2. Compute loss:      loss = criterion(y_hat, y)
3. Backward pass:     loss.backward()
4. Parameter update:  optimizer.step()
```

We train the XOR network from scratch — random initialisation, sigmoid activations, BCE loss, 1000 epochs.

In [ ]:
torch.manual_seed(42)

# XOR data
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

# 2-layer MLP: 2 → 4 → 1
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Sigmoid(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)

print('Training XOR network — 1000 epochs:')
print(f'  {"Epoch":>6}   {"Loss":>8}')
print(f'  {"-"*6}   {"-"*8}')
for epoch in range(1000):
    optimizer.zero_grad()
    y_hat = model(X)
    loss = criterion(y_hat, y)
    loss.backward()
    optimizer.step()
    if epoch % 200 == 0 or epoch == 999:
        print(f'  {epoch:>6}   {loss.item():>8.4f}')

print()
print('Final predictions:')
with torch.no_grad():
    preds = model(X)
    for x_i, p, t in zip(X, preds, y):
        print(f'  x={x_i.tolist()}  y_hat={p.item():.4f}  predicted={int(p.item() > 0.5)}  target={int(t.item())}')

---
## Part 6: L-Layer Deep Feedforward Network (Note 09)

Note 09 generalises the 2-layer MLP to an arbitrary L-layer network. In numpy (09a) we implemented this as a loop over weight matrices. In PyTorch, `nn.Sequential` with a list of `nn.Linear` and activation layers does the same thing — the loop is implicit.

We build the MNIST-sized architecture from Note 09 Part 6: [784, 512, 256, 128, 10], ReLU hidden layers, softmax output.

In [ ]:
def build_network(layer_sizes, hidden_activation=nn.ReLU):
    """Build a fully connected network from a list of layer sizes.
    Hidden layers use hidden_activation. Output layer is linear (no activation)."""
    layers = []
    for i in range(len(layer_sizes) - 1):
        layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]))
        if i < len(layer_sizes) - 2:   # not the output layer
            layers.append(hidden_activation())
    return nn.Sequential(*layers)

mnist_sizes = [784, 512, 256, 128, 10]
model_mnist = build_network(mnist_sizes)

print('MNIST architecture:')
print(model_mnist)
print()

total_params = sum(p.numel() for p in model_mnist.parameters())
print(f'Total parameters: {total_params:,}  (expected: 567,434)')

Run a forward pass on a single simulated input (one flattened 28×28 image) and verify the output shape and softmax sum.

In [ ]:
torch.manual_seed(0)
x_mnist = torch.rand(784)   # simulated pixel values in [0, 1]

with torch.no_grad():
    logits = model_mnist(x_mnist)
    probs  = torch.softmax(logits, dim=0)

print(f'Input shape:  {x_mnist.shape}')
print(f'Logits shape: {logits.shape}')
print(f'Probs shape:  {probs.shape}')
print()
print(f'Class probabilities: {probs.round(decimals=4).tolist()}')
print(f'Sum: {probs.sum().item():.6f}  (should be 1.0)')
print(f'Predicted class: {probs.argmax().item()}')

The same `build_network` function works for any architecture. We verify the deep-narrow vs shallow-wide comparison from Note 09 Part 3.

In [ ]:
arch_A = build_network([4, 8, 8, 8, 2])
arch_B = build_network([4, 64, 2])

params_A = sum(p.numel() for p in arch_A.parameters())
params_B = sum(p.numel() for p in arch_B.parameters())

print(f'Architecture A [4,8,8,8,2]:  {params_A} parameters  (expected: 202)')
print(f'Architecture B [4,64,2]:     {params_B} parameters  (expected: 450)')

---
## Summary

| What was demonstrated | numpy (Notes 01–08) | PyTorch equivalent |
|---|---|---|
| Perceptron forward pass | `np.dot(w, x) + b` | `torch.dot(w, x) + b` |
| MLP layer | `W.T @ x + b` | `nn.Linear(in, out)` |
| Activation functions | hand-coded formulas | `torch.sigmoid`, `torch.relu`, etc. |
| Loss functions | hand-coded formulas | `nn.MSELoss`, `nn.BCELoss`, `nn.CrossEntropyLoss` |
| Backward pass | chain rule by hand | `loss.backward()` |
| Weight update | `W -= eta * dW` | `optimizer.step()` |
| L-layer network | loop over weight list | `nn.Sequential` + `build_network` |
| Parameter count | manual formula | `sum(p.numel() for p in model.parameters())` |